<a href="https://colab.research.google.com/github/GulaiRandang/Data-Science_IF405_2026/blob/main/Pertemuan3_%5BAbdurrhman_AL_Asykari%5D_%5B240401010131%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================
# PIPELINE DATA CLEANING — HOUSING DATASET
# ============================================

# LANGKAH 2 — Import semua library yang diperlukan
import pandas as pd
import numpy as np
import scipy.stats.mstats as mstats
import requests  # Tambahan untuk akses API di Langkah 11

# LANGKAH 3 — Muat dataset
# Pastikan file 'housing_dirty.csv' sudah di-upload ke Google Colab
df = pd.read_csv('housing_dirty.csv')
print('Shape awal:', df.shape)

# LANGKAH 4 — Eksplorasi awal
print("\n--- Info Dataset ---")
df.info()
print("\n--- Deskripsi Dataset ---")
print(df.describe())
print("\n--- Jumlah Missing Values Per Kolom ---")
print(df.isnull().sum())

# LANGKAH 5 — Hapus baris duplikat
df.drop_duplicates(inplace=True)
print('\nSetelah hapus duplikat:', df.shape)

# LANGKAH 6 — Normalisasi string
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# LANGKAH 7 — Imputasi missing values
# Median untuk numerik
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
# Modus untuk kategorik (kamar/kondisi/kota tergantung tipe di dataset)
if 'kamar' in df.columns:
    df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

# LANGKAH 8 — Tangani outlier dengan IQR Fence
# Hanya untuk kolom harga_juta dan luas_m2 sesuai instruksi langkah 8
for col in ['harga_juta', 'luas_m2']:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_fence = Q1 - 1.5 * IQR
        upper_fence = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower_fence, upper_fence)

# LANGKAH 9 — Validasi akhir
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('\nShape akhir setelah validasi:', df.shape)

# LANGKAH 10 — Ekspor ke CSV lokal
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih "housing_clean.csv" berhasil tersimpan!')


# ============================================
# LANGKAH 11 — AKSES API JSONPLACEHOLDER
# ============================================
print("\n--- Mengakses API JSONPlaceholder ---")
url_api = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(url_api)

if response.status_code == 200:
    # Mengubah response JSON menjadi DataFrame
    df_api = pd.DataFrame(response.json())
    print("Berhasil memuat data dari API!")
    print("Shape data API:", df_api.shape)
    print(df_api.head())
else:
    print("Gagal mengakses API. Status code:", response.status_code)

Shape awal: (130, 7)

--- Info Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

--- Deskripsi Dataset ---
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    8